# 04 - Delta SQL Exploration

Query business KPIs and prove that each output is a real Delta table by inspecting transaction history.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.delta_pipeline import DeltaSettings, create_delta_spark_session, delta_history

settings = DeltaSettings.from_env()
spark = create_delta_spark_session("InsightsStream-Explore-Delta", settings)
gold = spark.read.format("delta").load(settings.uri("gold/daily_metrics"))
gold.createOrReplaceTempView("gold_daily_metrics")

In [ ]:
spark.sql("""
    SELECT metric, SUM(event_count) AS events, ROUND(SUM(total_value), 2) AS total_value
    FROM gold_daily_metrics
    GROUP BY metric
    ORDER BY total_value DESC
""").show(truncate=False)

for table in ("bronze/events", "silver/events", "gold/daily_metrics", "processed/events"):
    print(f"Delta history: {table}")
    delta_history(spark, table, settings).select("version", "timestamp", "operation").show(5, truncate=False)